In [2]:
pip install scikit-learn pandas joblib

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install pdfplumber pandas regex tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 18.7 MB/s  0:00:00 21.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 12.3 MB/s  0:00:004.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber] 3/5 [pdfminer.six]
Note: you may need to restart the kernel to use updated packages.


In [5]:
import pdfplumber

def extract_pdf_text(path: str) -> str:
    full_text = []

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text.append(text)

    return "\n".join(full_text)


In [6]:
import re
from typing import List, Dict


CASE_SPLIT_PATTERN = r"(BOA-\d+)"


def split_cases(text: str) -> List[str]:
    parts = re.split(CASE_SPLIT_PATTERN, text)

    cases = []
    for i in range(1, len(parts), 2):
        boa_number = parts[i]
        body = parts[i + 1]
        cases.append(f"{boa_number} {body}")

    return cases


def parse_case(case_text: str) -> Dict:

    # BOA number
    boa_match = re.search(r"(BOA-\d+)", case_text)
    boa_number = boa_match.group(1) if boa_match else None

    # Address
    address_match = re.search(r"(\d+.*?, Ward \d+)", case_text)
    address = address_match.group(1) if address_match else None

    # Ward
    ward_match = re.search(r"Ward (\d+)", case_text)
    ward = ward_match.group(1) if ward_match else None

    # Violations
    violations = re.findall(r"Article .*? - (.*?)\.", case_text)
    num_violations = len(violations)

    # Relief type
    relief_type = None
    if "Variance" in case_text:
        relief_type = "Variance"
    elif "Conditional Use" in case_text:
        relief_type = "Conditional Use"

    # Outcome
    if "Appeal Granted" in case_text or "Appeal Sustained" in case_text:
        outcome = "Granted"
    elif "Appeal Denied" in case_text:
        outcome = "Denied"
    else:
        outcome = "Unknown"

    # Vote
    vote_match = re.search(r"(\d+)-(\d+)", case_text)
    vote = vote_match.group(0) if vote_match else None

    return {
        "boa_number": boa_number,
        "address": address,
        "ward": ward,
        "num_violations": num_violations,
        "violations": "; ".join(violations),
        "relief_type": relief_type,
        "vote": vote,
        "outcome": outcome,
    }


In [1]:
import pandas as pd
from extract_text import extract_pdf_text
from parse_cases import split_cases, parse_case
import glob
from tqdm import tqdm


def build_dataset(pdf_folder: str):

    all_rows = []

    pdf_files = glob.glob(f"{pdf_folder}/*.pdf")

    for file in tqdm(pdf_files):
        text = extract_pdf_text(file)
        cases = split_cases(text)

        for case in cases:
            parsed = parse_case(case)
            all_rows.append(parsed)

    df = pd.DataFrame(all_rows)

    return df


if __name__ == "__main__":
    df = build_dataset("pdfs")
    df.to_csv("zba_cases_2026.csv", index=False)
    print("Dataset built.")


ModuleNotFoundError: No module named 'extract_text'